# Coffee Standard v8 — AF2 vs D0FT paired three-seed confirmation
Tidak melakukan training. Mengevaluasi checkpoint seed 42, 123, dan 2026 yang sudah ada.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import importlib, os, shutil, subprocess, sys, tarfile, torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-igem-paired-confirmation'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/coffee-detection-with-standard-v8-external-sni18-v1.tar',))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/coffee-detection-with-standard-v8-external-sni18-v1.tar')
DATA=Path('/content/coffee-standard-v8-external')
if DATA.exists(): shutil.rmtree(DATA)
with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
OUTPUT=PROJECT_ROOT/'experiments/coffee-standard-v8-af2-paired-v1'
print('GPU:',torch.cuda.get_device_name(0)); print('OUTPUT:',OUTPUT)


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_coffee_standard_af2_paired','--project-root',str(PROJECT_ROOT),'--data-root',str(DATA),'--output-root',str(OUTPUT),'--device','0']
LOG=OUTPUT/'run.log'; OUTPUT.mkdir(parents=True,exist_ok=True)
print('MENJALANKAN:',' '.join(command),flush=True)
with LOG.open('w',encoding='utf-8') as log:
    process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        log.write(line); log.flush()
        if line.startswith(('EVALUATE','REUSE')): print(line,end='',flush=True)
code=process.wait()
if code:
    print(''.join(LOG.read_text(errors='replace').splitlines(keepends=True)[-120:])); raise RuntimeError(f'Gagal: {code}')


In [ ]:
import json,pandas as pd
from IPython.display import display
summary=json.loads((OUTPUT/'coffee_standard_af2_paired_summary.json').read_text())
display(pd.DataFrame(summary['aggregate']).style.format({column:'{:.2%}' for column in ('d0ft_mean','d0ft_std','af2_mean','af2_std','delta_mean','delta_std','delta_min')}))
print('CRITERIA:',summary['criteria']); print('DECISION:',summary['decision']); print('TRAINING:',summary['training_executed'],'TEST:',summary['test_images_accessed'])
print('Kirim tabel dan keputusan.')
